In [1]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

In [2]:
from data_withdiffusion import get_dataset_withdiffusion
train_loader, val_loader, test_loader = get_dataset_withdiffusion(MODEL_PATH = '/tf/hsh/SW_ECG/RDDM_ECG/finalmodel/fftloss/1to', DATA_PATH = '/tf/hsh/SW_ECG/data/', only_one=False, lead_num=[4])#, lead_num=[2,3,4,5,6,7,8,9,10,11,12])#, no_diffusion=True)

Deterministic with seed = 31
physionet2017/RDDM
3 0


100%|██████████| 1684/1684 [1:38:36<00:00,  3.51s/it]


----data setting with diffusion 완료----


### CPSC2018

In [ ]:
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader, random_split

import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader, random_split
import sklearn.preprocessing as skp

def get_combined_dataloader_from_npy(
    DATA_PATH="/cap/RDDM-main/datasets/PTBXL/temp",
    lead_num=[1, 4],
    window_size=10,
    label_name="y",
    batch_size=16,
    stack_mode=True
):

    all_leads = []

    for lead in lead_num:
        # 각 리드별 train/test 불러오기
        test = np.load(f"{DATA_PATH}/lead{lead}_test.npy", allow_pickle=True).reshape(-1, 128 * window_size)

        full = np.concatenate([test], axis=0)  # (N, 1280)
        full = skp.minmax_scale(full, (-1, 1), axis=1)
        all_leads.append(full)

    # Stack or Concat
    if stack_mode:
        ecg_data = np.stack(all_leads, axis=1)  # (N, L, 1280)
    else:
        ecg_data = np.concatenate(all_leads, axis=1)  # (N, L*1280)

    # 라벨 불러오기
    labels_test = np.load(f"{DATA_PATH}/{label_name}_test.npy", allow_pickle=True)

    
    labels = np.concatenate([labels_test], axis=0)  # (N,)

    # Tensor 변환
    ecg_tensor = torch.tensor(ecg_data, dtype=torch.float32)
    label_tensor = torch.tensor(labels, dtype=torch.float32) - 1
    print(label_tensor, max(label_tensor), min(label_tensor))
    dataset = TensorDataset(ecg_tensor, label_tensor)

    # train/val/test 분할
    N = len(dataset)
    train_len = int(N * 0.6)
    val_len = int(N * 0.2)
    test_len = N - train_len - val_len

    train_set, val_set, test_set = random_split(dataset, [train_len, val_len, test_len])
    train_loader = DataLoader(train_set, batch_size=batch_size, shuffle=True)
    val_loader = DataLoader(val_set, batch_size=batch_size, shuffle=False)
    test_loader = DataLoader(test_set, batch_size=batch_size, shuffle=False)

    print(f"✅ Loaded leads {lead_num} with shape {ecg_tensor.shape}")
    return train_loader, val_loader, test_loader



In [ ]:
train_loader, val_loader, test_loader = get_combined_dataloader_from_npy(
    DATA_PATH="/tf/hsh/SW_ECG/data/CPSC2018/RDDM",
    lead_num=[1],  # 원하는 리드 조합
    window_size=10,
    batch_size=16,
    stack_mode=True
)

### CNN model

In [3]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score
import numpy as np
import matplotlib.pyplot as plt

class ECG_CNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=(3, 5), stride=1, padding=(1, 2)),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d((1, 2)),
            nn.Conv2d(32, 64, kernel_size=(3, 5), padding=(1, 2)),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.AdaptiveMaxPool2d((1, 10))
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 1 * 10, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = x.unsqueeze(1)  # (B, 1, 12, 1280)
        x = self.conv(x)
        return self.classifier(x)

ST-MEM model

In [4]:
import torch.optim as optim
from model import ST_MEM
import torch.nn as nn
import torch

class ECGFeatureClassifier(nn.Module):
    def __init__(self, model: ST_MEM, num_classes: int = 9, freeze_vit: bool = True):
        super().__init__()
        self.vit = model.encoder
        if freeze_vit:
            for param in self.vit.parameters():
                param.requires_grad = False 
        self.classifier = nn.Linear(768, num_classes)

    def forward(self, x, lead_num=1):
        x = self.vit.forward_encoding(x, lead_num)
        out = self.classifier(x)
        return out

st_mem = ST_MEM(seq_len = 1260,
                patch_size = 42,
                num_leads = 12,
                embed_dim = 768,
                depth = 12,
                num_heads = 12,
                decoder_embed_dim = 256,
                decoder_depth = 4,
                decoder_num_heads = 4,
                mlp_ratio = 4,
                qkv_bias = True,
                norm_layer = nn.LayerNorm,
                norm_pix_loss = False)
# checkpoint 로드
checkpoint = torch.load("/tf/hsh/SW_ECG/st_mem_128hz/st_mem_128.pth", map_location='cpu')
state_dict = checkpoint["model"]

st_mem.load_state_dict(state_dict)

# freeze 및 eval 모드
for param in st_mem.parameters():
    param.requires_grad = False
st_mem.eval()
device="cuda"
model = ECGFeatureClassifier(model=st_mem, num_classes=4, freeze_vit=True).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

### 실행 코드

In [5]:
from sklearn.metrics import classification_report
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torch.utils.tensorboard import SummaryWriter
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report
from tqdm import tqdm
import random
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import pandas as pd

def evaluate_with_report(model, loader, criterion, device, phase="Eval", class_names=None):
    model.eval()
    total_loss = 0
    preds_all, targets_all = [], []
    probs_all = []

    loop = tqdm(loader, desc=f"{phase} (with report)", leave=False)

    with torch.no_grad():
        for x, y in loop:
            x, y = x.to(device), y.to(device)
            output = model(x[:, :, :1260])
            loss = criterion(output, y.long())
            total_loss += loss.item() * x.size(0)
            preds_all.append(output.cpu())
            targets_all.append(y.cpu())
            probs_all.append(torch.softmax(output, dim=1).cpu())
            loop.set_postfix(loss=loss.item())

    preds = torch.cat(preds_all).argmax(dim=1).numpy()
    targets = torch.cat(targets_all).numpy()
    probs = torch.cat(probs_all).numpy()

    acc = accuracy_score(targets, preds)
    f1 = f1_score(targets, preds, average='macro')
    
    try:
        auc = roc_auc_score(targets, probs, multi_class='ovr')
    except ValueError:
        auc = float('nan')

    # 🔹 classification report (precision, recall, f1-score)
    report = classification_report(targets, preds, target_names=class_names, digits=4)
    print("\n📄 Classification Report:\n")
    print(report)

    # 🔹 per-class accuracy 계산
    cm = confusion_matrix(targets, preds)
    per_class_accuracy = cm.diagonal() / cm.sum(axis=1)

    if class_names is None:
        class_names = [f'class_{i}' for i in range(len(per_class_accuracy))]

    print("\n📊 Per-Class Accuracy:")
    for cls_name, acc_val in zip(class_names, per_class_accuracy):
        print(f"{cls_name:>10}: {acc_val:.4f}")

    # 🔹 결과 반환
    return total_loss / len(loader.dataset), acc, f1, auc, report, per_class_accuracy

2025-06-22 06:12:41.340515: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750572761.349111 2131997 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750572761.351578 2131997 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-22 06:12:41.361971: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


In [6]:
def train(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    loop = tqdm(loader, desc="Training", leave=False)
    
    for x, y in loop:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        #output = model(x)
        output = model(x[:,:,:1260])
        loss = criterion(output, y.long())
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * x.size(0)
        loop.set_postfix(loss=loss.item())
    return total_loss / len(loader.dataset)

def evaluate(model, loader, criterion, device, phase="Eval"):
    model.eval()
    total_loss = 0
    preds_all, targets_all = [], []
    probs_all = []
    loop = tqdm(loader, desc=phase, leave=False)
    
    with torch.no_grad():
        for x, y in loop:
            x, y = x.to(device), y.to(device)
            #output = model(x)
            output = model(x[:,:,:1260])
            loss = criterion(output, y.long())
            total_loss += loss.item() * x.size(0)
            preds_all.append(output.cpu())
            targets_all.append(y.cpu())
            probs_all.append(torch.softmax(output, dim=1).cpu())
            loop.set_postfix(loss=loss.item())
    preds = torch.cat(preds_all).argmax(dim=1)
    targets = torch.cat(targets_all)
    probs = torch.cat(probs_all)
    
    acc = accuracy_score(targets, preds)
    f1 = f1_score(targets, preds, average='macro')
    
    try:
        auc = roc_auc_score(torch.cat(targets_all), probs, multi_class='ovr')
    except ValueError:
        auc = float('nan')  # 클래스가 하나만 나왔을 경우 등
    
    return total_loss / len(loader.dataset), acc, f1, auc

def load_ecg_leads(prefix, type, num_leads=12, window_size = 10):
        leads = []
        for i in range(1, num_leads + 1):
            lead = np.load(f"{prefix}lead{i}_{type}.npy")  # e.g., lead1_train.npy
            lead = lead.reshape(-1, window_size * 128)
            leads.append(lead)
        return np.stack(leads, axis=1)  # shape: (N, 12, 1280)

class EarlyStopping:
    def __init__(self, patience=5, verbose=True, delta=0.0):
        self.patience = patience
        self.verbose = verbose
        self.delta = delta
        self.best_loss = float('inf')
        self.counter = 0
        self.early_stop = False
        self.best_model_state = None

    def __call__(self, val_loss, model):
        if val_loss < self.best_loss - self.delta:
            self.best_loss = val_loss
            self.counter = 0
            self.best_model_state = model.state_dict()
            if self.verbose:
                print(f"Validation loss improved. Resetting counter.")
        else:
            self.counter += 1
            if self.verbose:
                print(f"No improvement. Patience {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)  # GPU용 시드

    # CuDNN 관련 설정 (완전한 재현성 위해)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

# 4. 실행
def model_test(is_cnn=True, epochs=51):
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    
    criterion = nn.CrossEntropyLoss()

    # 모델, 손실함수, 옵티마이저
    
    if is_cnn :
        model = ECG_CNN(num_classes=4).to(device)
    else :
        model = ECGFeatureClassifier(model=st_mem, num_classes=4, freeze_vit=True).to(device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-4)

    # TensorBoard
    writer = SummaryWriter(log_dir="runs/exp4")

    early_stopper = EarlyStopping(patience=7)

    # 학습 루프
    for epoch in range(1, epochs):
        train_loss = train(model, train_loader, criterion, optimizer, device)
        val_loss, val_acc, val_f1, val_auc = evaluate(model, val_loader, criterion, device, phase='Val')
        early_stopper(val_loss, model)
        print(f"[Epoch {epoch}] Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Acc: {val_acc:.4f} | F1: {val_f1:.4f} | AUC: {val_auc:.4f}")
        writer.add_scalar("AUC/val", val_auc, epoch)
        writer.add_scalar("Loss/train", train_loss, epoch)
        writer.add_scalar("Loss/val", val_loss, epoch)
        writer.add_scalar("Accuracy/val", val_acc, epoch)
        writer.add_scalar("F1/val", val_f1, epoch)
        if early_stopper.early_stop:
            print("Early stopping triggered.")
            break
    
    torch.save(early_stopper.best_model_state, 'best_model.pt')
    # # 테스트 평가
    model.load_state_dict(early_stopper.best_model_state)
    # 나중에 불러오기 (inference 시점)
    
    if is_cnn :
        model = ECG_CNN(num_classes=4).to(device)
        #model = SignalClassifier1DCNN(num_channels=5, num_classes=5).to(device)
    else :
        model = ECGFeatureClassifier(model=st_mem, num_classes=4, freeze_vit=True).to(device)
    
    model.load_state_dict(torch.load('best_model.pt'))

    class_names = ["NORM", "AF", "I-AVB", "LBBB", "RBBB", "PAC", "PVC", "STD", "STE"]
    
    test_loss, test_acc, test_f1, test_auc, report_str, class_accs = evaluate_with_report(
        model, test_loader, criterion, device,
        phase="Test", class_names=class_names
    )
    
    print(f"\n✅ [FINAL TEST RESULT] Loss: {test_loss:.4f} | Acc: {test_acc:.4f} | F1: {test_f1:.4f} | AUC: {test_auc:.4f}")

    # test_loss, test_acc, test_f1, test_auc = evaluate(model, test_loader, criterion, device)
    # print(f"\n[TEST] Loss: {test_loss:.4f} | Acc: {test_acc:.4f} | F1: {test_f1:.4f} | AUC: {test_auc:.4f}")

In [ ]:
model_test(True, epochs=81)

In [7]:
model_test(False, epochs=151)

Validation loss improved. Resetting counter.
[Epoch 1] Train Loss: 0.9528 | Val Loss: 0.9099 | Acc: 0.6054 | F1: 0.2545 | AUC: 0.7274


Validation loss improved. Resetting counter.
[Epoch 2] Train Loss: 0.8929 | Val Loss: 0.8840 | Acc: 0.6154 | F1: 0.2901 | AUC: 0.7421


Validation loss improved. Resetting counter.
[Epoch 3] Train Loss: 0.8730 | Val Loss: 0.8705 | Acc: 0.6225 | F1: 0.3133 | AUC: 0.7512


Validation loss improved. Resetting counter.
[Epoch 4] Train Loss: 0.8611 | Val Loss: 0.8617 | Acc: 0.6294 | F1: 0.3327 | AUC: 0.7582


Validation loss improved. Resetting counter.
[Epoch 5] Train Loss: 0.8528 | Val Loss: 0.8551 | Acc: 0.6325 | F1: 0.3418 | AUC: 0.7637


...

Validation loss improved. Resetting counter.
[Epoch 147] Train Loss: 0.7593 | Val Loss: 0.7855 | Acc: 0.6494 | F1: 0.4069 | AUC: 0.8129


Validation loss improved. Resetting counter.
[Epoch 148] Train Loss: 0.7592 | Val Loss: 0.7854 | Acc: 0.6490 | F1: 0.4068 | AUC: 0.8129


Validation loss improved. Resetting counter.
[Epoch 149] Train Loss: 0.7590 | Val Loss: 0.7853 | Acc: 0.6488 | F1: 0.4067 | AUC: 0.8130


Validation loss improved. Resetting counter.
[Epoch 150] Train Loss: 0.7588 | Val Loss: 0.7853 | Acc: 0.6492 | F1: 0.4071 | AUC: 0.8130


ValueError: Number of classes, 4, does not match size of target_names, 9. Try specifying the labels parameter